In [7]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def load_metadata_json(path, species_label):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    for gse_id, meta in data.items():
        title = meta.get("Title", "")
        summary = meta.get("Summary", "")
        text = f"{title} {summary}".strip()

        rows.append({
            "gse_id": gse_id,
            "species": species_label,
            "title": title,
            "summary": summary,
            "text": text
        })

    return pd.DataFrame(rows)


# 1. load data
human_df = load_metadata_json("metadata/metadata_human.json", "human")
mouse_df = load_metadata_json("metadata/metadata_mouse.json", "mouse")

# 2. combine for shared TF-IDF vocabulary
all_df = pd.concat([human_df, mouse_df], ignore_index=True)

In [8]:
# 3. TF-IDF representation
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(all_df["text"])

# 4. split human / mouse matrix
n_human = len(human_df)

X_human = X[:n_human]
X_mouse = X[n_human:]

In [9]:
# 5. calculate all human-mouse similarities
sim_matrix = cosine_similarity(X_human, X_mouse)

# 6. rank all pairs
rows = []

for i in range(sim_matrix.shape[0]):
    for j in range(sim_matrix.shape[1]):
        rows.append({
            "human_gse": human_df.iloc[i]["gse_id"],
            "mouse_gse": mouse_df.iloc[j]["gse_id"],
            "similarity": sim_matrix[i, j],
            "human_title": human_df.iloc[i]["title"],
            "mouse_title": mouse_df.iloc[j]["title"],
        })

pairs_df = pd.DataFrame(rows)

top_pairs = pairs_df.sort_values("similarity", ascending=False)

top_pairs.head(30)

KeyboardInterrupt: 

In [2]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["text"])

sim_matrix = cosine_similarity(X)

In [ ]:
import numpy as np
import pandas as pd

top_n = 100

# flatten similarity matrix
flat = sim_matrix.ravel()

# get indices of top N similarities
top_indices = np.argpartition(flat, -top_n)[-top_n:]
top_indices = top_indices[np.argsort(flat[top_indices])[::-1]]

rows = []

for idx in top_indices:
    i, j = np.unravel_index(idx, sim_matrix.shape)

    rows.append({
        "human_gse": human_df.iloc[i]["gse_id"],
        "mouse_gse": mouse_df.iloc[j]["gse_id"],
        "similarity": sim_matrix[i, j],
        "human_title": human_df.iloc[i]["title"],
        "mouse_title": mouse_df.iloc[j]["title"],
    })

top_pairs = pd.DataFrame(rows)

top_pairs

In [ ]:
top_pairs.to_csv("top_human_mouse_tfidf_pairs.csv", index=False)

In [ ]:
human_sim = cosine_similarity(X_human)
mouse_sim = cosine_similarity(X_mouse)